# Fan Chart Underwriting

**Goal:** Use the full P10/P90 probability fan chart to model upside, base, and downside acquisition scenarios for a specific parcel.

Unlike point-estimate AVMs, Homecastr gives you the *distribution* — enabling proper scenario analysis at acquisition.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/homecastr/homecastr-cookbook/blob/main/examples/investment/03_fan_chart_underwriting.ipynb)

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from homecastr import HomecastrClient

client = HomecastrClient(os.getenv("HOMECASTR_API_KEY", "hc_demo_public_readonly"))

In [ ]:
# HCAD parcel ID for a Houston property.
# Find parcel IDs at: https://hcad.org/property-search/
ACCT = "0420430060006"

fan_df = client.forecast.by_parcel.retrieve_fan_chart(ACCT)

purchase_price = fan_df.attrs["current_value"]
print(f"Parcel:         {fan_df.attrs['acct']}")
print(f"Current value:  ${purchase_price:,}")
print(f"Origin year:    {fan_df.attrs['origin_year']}")
print()
print(fan_df.to_string(index=False))

In [ ]:
# Simple IRR calculation for each scenario
def irr(purchase, exit_value, hold_years, annual_rent, vacancy=0.05, expenses=0.35):
    """Approximate unlevered IRR."""
    noi = annual_rent * (1 - vacancy) * (1 - expenses)
    cash_flows = [-purchase] + [noi] * (hold_years - 1) + [noi + exit_value]
    # Newton's method for IRR
    r = 0.1
    for _ in range(100):
        npv = sum(cf / (1 + r) ** t for t, cf in enumerate(cash_flows))
        dnpv = sum(-t * cf / (1 + r) ** (t + 1) for t, cf in enumerate(cash_flows))
        if abs(dnpv) < 1e-10:
            break
        r -= npv / dnpv
    return r


ANNUAL_RENT = 28_800   # $2,400/month
HOLD_YEARS = 5

exit_row = fan_df[fan_df["year"] == fan_df["year"].max()].iloc[0]

scenarios = {
    "Downside (P10)": exit_row["p10"],
    "Base case (P50)": exit_row["p50"],
    "Upside (P90)": exit_row["p90"],
}

print(f"{'Scenario':<22} {'Exit Value':>12} {'IRR':>8} {'Equity Multiple':>16}")
print("-" * 60)
for label, exit_val in scenarios.items():
    r = irr(purchase_price, exit_val, HOLD_YEARS, ANNUAL_RENT)
    equity_mult = (exit_val + ANNUAL_RENT * HOLD_YEARS * 0.65) / purchase_price
    print(f"{label:<22} ${exit_val:>11,.0f} {r*100:>7.1f}% {equity_mult:>14.2f}x")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: fan chart
ax = axes[0]
ax.fill_between(fan_df["year"], fan_df["p10"] / 1e3, fan_df["p90"] / 1e3,
                alpha=0.15, color="steelblue", label="P10–P90")
ax.fill_between(fan_df["year"], fan_df["p25"] / 1e3, fan_df["p75"] / 1e3,
                alpha=0.3, color="steelblue", label="P25–P75")
ax.plot(fan_df["year"], fan_df["p50"] / 1e3, color="steelblue", lw=2, label="P50")
ax.axhline(purchase_price / 1e3, color="gray", ls="--", lw=1, label="Purchase price")
ax.set_title(f"Forecast Fan Chart — Parcel {ACCT}")
ax.set_xlabel("Year")
ax.set_ylabel("Value ($000s)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.0f}k"))
ax.legend()

# Right: IRR by scenario
ax2 = axes[1]
scenario_labels = list(scenarios.keys())
irr_values = [irr(purchase_price, v, HOLD_YEARS, ANNUAL_RENT) * 100 for v in scenarios.values()]
bar_colors = ["lightcoral", "steelblue", "mediumseagreen"]
bars = ax2.bar(scenario_labels, irr_values, color=bar_colors, edgecolor="k", linewidth=0.5)
ax2.axhline(8, color="red", ls="--", lw=1, label="8% hurdle rate")
ax2.set_ylabel("Unlevered IRR (%)")
ax2.set_title(f"5-Year IRR by Scenario\n(purchase: ${purchase_price:,.0f})")
for bar, val in zip(bars, irr_values):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
             f"{val:.1f}%", ha="center", va="bottom", fontweight="bold")
ax2.legend()

plt.tight_layout()
plt.show()